In [7]:
import json
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain_core.messages import (
    AIMessage,
    BaseMessage,
    HumanMessage,
    ToolMessage,
    SystemMessage,
)

In [8]:
# Load environment variables from .env file
load_dotenv()

# Setup LLM
code_generation_agent = ChatOpenAI(model="gpt-5", temperature=0.0, openai_api_key=os.getenv("OPENAI_API_KEY"))

In [9]:
def generate_questions(data):
    system_prompt = SystemMessage(
        content="You are a molecular dynamics expert and trying to generate questions related to interatomic potentials used in LAMMPS. You will be provided with a list of jsons containing details about interatomic potentials in form of notes and abstracts. Generate questions that can be answered using the provided information. The questions will present a specific scenario or problem related to molecular dynamics simulations and interatomic potentials that will be used in the particular scenario. The questions should be clear, concise, and relevant to the provided information. Avoid overly broad or vague questions. Focus on practical applications and implications of using different interatomic potentials in LAMMPS simulations. The questions should be designed to test the understanding of the material and encourage critical thinking about the selection and use of interatomic potentials in molecular dynamics simulations. The format of the output should be a json with two keys: question and potential. The potential key should contain the name of the potential that is most relevant to the question."
    )
    system_prompt.content += f" Here is the data: {data}"
    human_message = HumanMessage(content="Generate 15 questions.")
    
    response = code_generation_agent.invoke([system_prompt, human_message])
    return response.content

In [12]:
for element in ["Al", "C", "Be", "Si", "Na", "K", "Ca"]:
    with open(f"PotentialData/nist_potentials_{element}.json", "r") as f:
        data = json.load(f)
    questions = generate_questions(data)
    with open(f"Questions/generated_questions_{element}_1.json", "w") as f:
        f.write(questions)
    print(f"Generated questions for {element} and saved to generated_questions_{element}.json")

Generated questions for C and saved to generated_questions_C.json
Generated questions for Be and saved to generated_questions_Be.json
Generated questions for Si and saved to generated_questions_Si.json
Generated questions for Na and saved to generated_questions_Na.json
Generated questions for K and saved to generated_questions_K.json
Generated questions for Ca and saved to generated_questions_Ca.json


In [3]:
re_write_llm = ChatOpenAI(temperature=0, model_name="gpt-5", openai_api_key=os.getenv("OPENAI_API_KEY"))

# Create a prompt template for query rewriting
query_rewrite_template = """You are an AI assistant tasked with reformulating user queries to improve retrieval in a RAG system. 
Given the original query, rewrite it to be more specific, detailed, and likely to retrieve relevant information.

Original query: {original_query}

Rewritten query:"""

query_rewrite_prompt = PromptTemplate(
    input_variables=["original_query"],
    template=query_rewrite_template
)

# Create an LLMChain for query rewriting
query_rewriter = query_rewrite_prompt | re_write_llm

def rewrite_query(original_query):
    """
    Rewrite the original query to improve retrieval.
    
    Args:
    original_query (str): The original user query
    
    Returns:
    str: The rewritten query
    """
    response = query_rewriter.invoke(original_query)
    return response.content

In [ ]:
for element in ["Be", "Si", "Na", "K", "Ca"]:
    with open(f"Questions/generated_questions_{element}.json", "r") as f:
        questions = json.load(f)
    
    rewritten_questions = []
    for item in questions:
        print(f"Rewriting question: {item['question']}")
        original_question = item["question"]
        rewritten_question = rewrite_query(original_question)
        rewritten_questions.append({
            "question": rewritten_question,
            "potential": item["potential"]
        })
    
    with open(f"rewritten_questions_{element}.json", "w") as f:
        json.dump(rewritten_questions, f, indent=4)
    
    print(f"Rewritten questions for {element} and saved to rewritten_questions_{element}.json")